# Bài 2: Tạo & quản lý bảng Delta

## Mục tiêu
- Phân biệt bảng **managed** và **external** trong Delta Lake.
- Biết 3 cách tạo bảng: `CREATE TABLE` (SQL DDL), `DataFrameWriter.saveAsTable`, `DeltaTable.create` (Python API).
- Biết cách xem/​chỉnh metadata, table properties.


## 2.1. Managed table vs External table

| | **Managed table** | **External table** |
|---|---|---|
| Location | Do metastore quản lý (`spark.sql.warehouse.dir/<db>.db/<table>`) | Bạn tự chỉ định qua `LOCATION` |
| `DROP TABLE` | Xoá **cả metadata lẫn dữ liệu vật lý** | Chỉ xoá metadata, **dữ liệu vẫn còn** trên storage |
| Khi dùng | Bảng "sở hữu" hoàn toàn bởi 1 pipeline/team | Dữ liệu được nhiều hệ thống chia sẻ, hoặc muốn kiểm soát thủ công vòng đời file |

Cả hai loại đều đăng ký trong **Hive Metastore** (ở đây là `thrift://hive-metastore:9083`) nên đều truy vấn được qua tên `db.table` — khác biệt duy nhất là ai chịu trách nhiệm xoá dữ liệu vật lý.

## 2.2. Ba cách tạo bảng Delta

1. **SQL DDL**:
```sql
CREATE TABLE db.tbl (id INT, name STRING) USING DELTA;                 -- managed
CREATE TABLE db.tbl (id INT, name STRING) USING DELTA LOCATION 's3a://...'; -- external
```
2. **DataFrameWriter**:
```python
df.write.format("delta").mode("overwrite").saveAsTable("db.tbl")        # managed
df.write.format("delta").mode("overwrite").save("s3a://bucket/path")    # path-based, chưa đăng ký metastore
```
3. **`DeltaTable` Python API** (`delta.tables.DeltaTable`) — dùng khi cần thao tác lập trình (`.create()`, `.createIfNotExists()`), hữu ích khi cần builder pattern cho constraint/generated column phức tạp.

Một bảng cũng có thể tồn tại **chỉ dưới dạng path** (`s3a://.../my_table`, ghi bằng `.save()`) mà không đăng ký vào Hive Metastore — vẫn là 1 bảng Delta hợp lệ, đọc bằng `spark.read.format("delta").load(path)`, chỉ là không có tên `db.table` để query bằng SQL.


## 0. Thiết lập môi trường

Notebook này chạy trong container `spark-master` (Jupyter Lab, xem `startup.sh`), kết nối tới:
- **Spark cluster**: `spark://spark-master:7077`
- **Hive Metastore**: `thrift://hive-metastore:9083` (dùng làm catalog)
- **MinIO** (S3-compatible): dữ liệu bảng managed nằm dưới `s3a://data-platform/managed/...`

Image `deltaio/delta-docker` đã cấu hình sẵn Delta Lake trong `spark-defaults.conf` nên không cần khai báo `spark.jars.packages` mỗi lần tạo `SparkSession`.

Yêu cầu: `docker compose up -d hive-metastore minio spark-master spark-worker` đã chạy trước khi mở notebook này.


In [ ]:
from pyspark.sql import SparkSession

DB_NAME = "bai02"
WAREHOUSE_DIR = "s3a://data-platform/managed"

spark = (
    SparkSession.builder
    .appName("bai02-tao-bang")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-master")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} LOCATION '{WAREHOUSE_DIR}/{DB_NAME}.db'")
spark.sql(f"USE {DB_NAME}")
spark


## 2.3. Ví dụ minh hoạ

In [ ]:
# Cách 1: SQL DDL - managed table
spark.sql("DROP TABLE IF EXISTS bai02.customers_sql")
spark.sql("""
CREATE TABLE bai02.customers_sql (
    customer_id INT,
    full_name   STRING,
    country     STRING,
    signup_date DATE
)
USING DELTA
TBLPROPERTIES ('delta.appendOnly' = 'false', 'description' = 'Bảng khách hàng - tạo bằng DDL')
""")
spark.sql("DESCRIBE TABLE EXTENDED bai02.customers_sql").show(50, truncate=False)


In [ ]:
# Cách 2: DataFrameWriter.saveAsTable - managed table, partition theo country
spark.sql("DROP TABLE IF EXISTS bai02.customers_df")

from datetime import date
rows = [
    (1, "An Nguyen", "VN", date(2023, 1, 10)),
    (2, "Bao Tran", "VN", date(2023, 3, 5)),
    (3, "John Smith", "US", date(2022, 11, 20)),
]
df = spark.createDataFrame(rows, ["customer_id", "full_name", "country", "signup_date"])
(
    df.write.format("delta")
    .mode("overwrite")
    .partitionBy("country")
    .saveAsTable("bai02.customers_df")
)
spark.sql("SHOW PARTITIONS bai02.customers_df").show()


In [ ]:
# Cách 3: DeltaTable Python API - tao bang rong voi schema tuong minh, roi insert
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark.sql("DROP TABLE IF EXISTS bai02.customers_api")

(
    DeltaTable.createIfNotExists(spark)
    .tableName("bai02.customers_api")
    .addColumn("customer_id", "INT")
    .addColumn("full_name", "STRING")
    .addColumn("country", "STRING")
    .property("description", "Bảng tạo bằng DeltaTable API")
    .execute()
)
spark.sql("INSERT INTO bai02.customers_api VALUES (1, 'Le Van C', 'VN')")
spark.sql("SELECT * FROM bai02.customers_api").show()


In [ ]:
# External table: tu chi dinh LOCATION, DROP TABLE khong xoa du lieu vat ly
EXT_LOCATION = "s3a://data-platform/external/bai02_customers_ext"

df.write.format("delta").mode("overwrite").save(EXT_LOCATION)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS bai02.customers_ext
USING DELTA
LOCATION '{EXT_LOCATION}'
""")
spark.sql("DESCRIBE EXTENDED bai02.customers_ext").filter("col_name = 'Location'").show(truncate=False)

spark.sql("DROP TABLE bai02.customers_ext")
# Du lieu van con tren MinIO du bang da bi xoa khoi metastore:
print(spark.read.format("delta").load(EXT_LOCATION).count(), "dòng vẫn còn trên storage")


## 2.4. Thực hành

**Bài 1** — Tạo bảng managed `bai02.orders` bằng SQL DDL với các cột `(order_id INT, product STRING, qty INT, order_date DATE)`.

**Bài 2** — Tạo bảng managed `bai02.orders_partitioned` giống hệt schema trên nhưng **partition theo `order_date`**, dùng `DataFrameWriter`. Ghi ít nhất 3 dòng thuộc 2 ngày khác nhau. Dùng `SHOW PARTITIONS` để kiểm tra.

**Bài 3** — Tạo một bảng **external** tại path `s3a://data-platform/external/bai02_orders_ext`, sau đó `DROP TABLE`. Chứng minh bằng code rằng dữ liệu vẫn đọc được qua `spark.read.format("delta").load(...)`.

**Bài 4** — Dùng `DeltaTable.createIfNotExists` tạo bảng `bai02.orders_api` có ràng buộc: cột `qty` kiểu `INT`, thêm table property `'quality' = 'bronze'`. Dùng `DESCRIBE TABLE EXTENDED` để xác nhận property đã được lưu.

**Bài 5 (tư duy)** — Trong pipeline ETL nhiều tầng (bronze → silver → gold), theo bạn tầng nào nên dùng external table, tầng nào nên dùng managed table? Giải thích ngắn gọn.


### Vùng làm bài — Bài 1

In [ ]:
# TODO: Bài 1


### Vùng làm bài — Bài 2

In [ ]:
# TODO: Bài 2


### Vùng làm bài — Bài 3

In [ ]:
# TODO: Bài 3


### Vùng làm bài — Bài 4

In [ ]:
# TODO: Bài 4


### Vùng làm bài — Bài 5

_Viết câu trả lời của bạn ở đây._

---
## Gợi ý / đáp án tham khảo

In [ ]:
# Đáp án Bài 1
spark.sql("DROP TABLE IF EXISTS bai02.orders")
spark.sql("""
CREATE TABLE bai02.orders (
    order_id INT, product STRING, qty INT, order_date DATE
) USING DELTA
""")


In [ ]:
# Đáp án Bài 2
from datetime import date
spark.sql("DROP TABLE IF EXISTS bai02.orders_partitioned")
rows = [(1, "A", 2, date(2024,1,1)), (2, "B", 1, date(2024,1,1)), (3, "C", 5, date(2024,1,2))]
odf = spark.createDataFrame(rows, ["order_id", "product", "qty", "order_date"])
odf.write.format("delta").partitionBy("order_date").saveAsTable("bai02.orders_partitioned")
spark.sql("SHOW PARTITIONS bai02.orders_partitioned").show()


In [ ]:
# Đáp án Bài 3
EXT2 = "s3a://data-platform/external/bai02_orders_ext"
odf.write.format("delta").mode("overwrite").save(EXT2)
spark.sql(f"CREATE TABLE IF NOT EXISTS bai02.orders_ext USING DELTA LOCATION '{EXT2}'")
spark.sql("DROP TABLE bai02.orders_ext")
print(spark.read.format("delta").load(EXT2).count(), "dòng - dữ liệu vẫn còn dù đã DROP TABLE")


In [ ]:
# Đáp án Bài 4
spark.sql("DROP TABLE IF EXISTS bai02.orders_api")
(
    DeltaTable.createIfNotExists(spark)
    .tableName("bai02.orders_api")
    .addColumn("order_id", "INT")
    .addColumn("qty", "INT")
    .property("quality", "bronze")
    .execute()
)
spark.sql("DESCRIBE TABLE EXTENDED bai02.orders_api").filter("col_name like '%quality%' or col_name = 'Table Properties'").show(truncate=False)


**Đáp án Bài 5**: Tầng **bronze** (raw, gần như append-only, thường được nhiều pipeline/hệ thống khác cùng đọc, hoặc cần kiểm soát chặt vòng đời file để tránh mất dữ liệu khi có ai đó lỡ tay `DROP TABLE`) hợp với **external table**. Tầng **silver/gold** (do chính team/pipeline sở hữu, là kết quả trung gian/cuối có thể tái tạo lại từ bronze) hợp với **managed table** vì đơn giản hoá vòng đời — xoá bảng là xoá sạch, không để lại rác trên storage.
